<a href="https://colab.research.google.com/github/MihaiDogariu/CV3/blob/main/laborator/CV_3_Lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificarea imaginilor
Acest proiect implementeaza arhitectura AlexNet pentru clasificarea imaginilor reale. Baza de date aleasa pentru demonstratie este CIFAR10.

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets
from torchvision import transforms
from torch.utils.data.sampler import SubsetRandomSampler
from tqdm import tqdm


# Alegem configuratia sistemului (cpu/gpu)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Dimensiunea batch-ului - folosita atat la crearea dataloaders, cat si la antrenare
batch_size = 64

# Numarul de procese care incarca datele in paralel. Redimensionarea 32x32 -> 227x227 se face pe CPU,
# iar fara procese auxiliare GPU-ul asteapta dupa date. Setati 0 daca apar probleme la incarcarea datelor.
num_workers = 4

## 1. Pre-procesarea datelor

In [ ]:
def get_loaders(data_dir,
               batch_size,
               augment,
               random_seed,
               valid_size=0.1,
               shuffle=True,
               num_workers=0):

    # Fiind o baza de date foarte populara, CIFAR10 poate fi descarcata cu ajutorul modulului torchvision
    # Transformarile se stabilesc mai jos, dupa ce calculam statisticile pentru normalizare
    train_dataset = datasets.CIFAR10(root=data_dir,
                                     train=True,
                                     download=True,
                                     )

    # Alegem numarul de esantioane pentru train/val
    num_train = len(train_dataset)
    indices = list(range(num_train))
    split = int(np.floor(valid_size * num_train))

    # Amestecam indecsii
    if shuffle:
        np.random.seed(random_seed)
        np.random.shuffle(indices)

    # Separam indecsii de train in train+val (baza de test este separata si se foloseste integral)
    train_idx, valid_idx = indices[split:], indices[:split]
    train_sampler = SubsetRandomSampler(train_idx)
    valid_sampler = SubsetRandomSampler(valid_idx)

    # Calculam media si deviatia standard pentru normalizarea bazei de date - acestea sunt calculate la nivel de canal si doar pe imaginile de antrenare (fara val si test)!
    # .data contine imaginile ca tablou uint8 de forma (N, 32, 32, 3); le aducem in [0, 1], ca dupa ToTensor()
    train_data = torch.from_numpy(train_dataset.data[train_idx]).float().div_(255)
    std, mean = torch.std_mean(train_data, dim=(0, 1, 2)) # statistici pe toate imaginile si pixelii, separat pentru fiecare canal (R, G, B)
    del train_data # eliberam memoria (~550 MB)
    print('Normalization computed on {} training images: mean = {}, std = {}'.format(len(train_idx), mean.tolist(), std.tolist()))

    normalize = transforms.Normalize(mean=mean.tolist(), std=std.tolist())

    # Definim setul de transformari necesare bazei de date
    valid_transform = transforms.Compose([
            transforms.Resize((227,227)), # baza de date CIFAR10 contine imagini de dimensiunea 32x32, iar AlexNet are intrari de dimensiune 227x227
            transforms.ToTensor(),        # transformarea intrarilor in tensori
            normalize,                    # aplicarea normalizarii
    ])
    if augment:
        train_transform = transforms.Compose([
            transforms.RandomCrop(32, padding=4), # decuparea unor regiuni aleatoare de dimensiune 32x32 din imaginea originala la care s-a adaugat padding=4
            transforms.RandomHorizontalFlip(0.4), # oglindirea imaginilor cu probabilitate de 40%
            transforms.Resize((227,227)),         # redimensionarea imaginilor augmentate la dimensiunea de 227x227 pixeli
            transforms.ToTensor(),                # transformarea intrarilor in tensori
            normalize,                            # aplicarea normalizarii
        ])
    else:
        train_transform = valid_transform

    # Transformari identice cu cele pentru validare. Normalizarea se face cu aceleasi valori ca in cazul train!
    test_transform = valid_transform

    train_dataset.transform = train_transform

    valid_dataset = datasets.CIFAR10(root=data_dir,
                                     train=True,
                                     download=True,
                                     transform=valid_transform,
                                     )

    # Descarcarea bazei de test
    test_dataset = datasets.CIFAR10(root=data_dir,
                                    train=False,
                                    download=True,
                                    transform=test_transform,
                                    )

    # Cream dataloaders pentru train, val si test
    # persistent_workers pastreaza procesele de incarcare intre epoci (pornirea lor este lenta, mai ales pe Windows)
    # pin_memory accelereaza copierea batch-urilor pe GPU
    loader_args = dict(batch_size=batch_size,
                       num_workers=num_workers,
                       persistent_workers=num_workers > 0,
                       pin_memory=torch.cuda.is_available())

    train_loader = torch.utils.data.DataLoader(
        train_dataset, sampler=train_sampler, **loader_args)

    valid_loader = torch.utils.data.DataLoader(
        valid_dataset, sampler=valid_sampler, **loader_args)

    # La evaluare nu este nevoie de amestecarea datelor
    test_loader = torch.utils.data.DataLoader(
        test_dataset, shuffle=False, **loader_args)

    return (train_loader, valid_loader, test_loader)


# Crearea efectiva a dataloaders
train_loader, valid_loader, test_loader = get_loaders(
    data_dir = './data',
    batch_size = batch_size,
    augment = True,
    random_seed = 1,
    num_workers = num_workers
)

## 2. Definirea modelului

Resurse utile (documentatii):
- strat convolutional: https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
- strat complet conectat: https://pytorch.org/docs/stable/generated/torch.nn.Linear.html?highlight=linear#torch.nn.Linear
- strat max pooling: https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html?highlight=maxpool#torch.nn.MaxPool2d
- activare ReLU: https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html?highlight=relu#torch.nn.ReLU
- regularizare dropout: https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html?highlight=dropout#torch.nn.Dropout
- mod secvential de compunere a straturilor: https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html?highlight=sequential#torch.nn.Sequential

In [ ]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super(AlexNet, self).__init__()
        # Intrare: 3x227x227
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=0),   # -> 96x55x55
            nn.BatchNorm2d(96),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2))                   # -> 96x27x27

        # TODO: completati restul straturilor

        self.fc3 = nn.Linear(4096, num_classes)

    def forward(self, x):
        out = self.layer1(x)

        # TODO: completati restul apelurilor straturilor definite in __init__()

        out = self.fc3(out)
        return out

## 3. Antrenarea retelei
In timpul antrenarii, testarea se face pe `val`. Acest bloc se ruleaza in mod repetat atunci cand optimizam modelul (schimbarea arhitecturii, schimbarea hiper-parametrilor, etc.). Nu se trece la pasul 4 decat dupa inghetarea starii modelului - dupa ce am terminat de optimizat.

In [ ]:
# Alegerea hiperparametrilor (batch_size este definit la inceput, deoarece este folosit la crearea dataloaders)
num_classes = 10
num_epochs = 10
learning_rate = 0.005

# Trecerea modelului pe gpu
model = AlexNet(num_classes).to(device)

# Alegerea functiei de pierdere. Clasificare de imagini => cross-entropy
criterion = nn.CrossEntropyLoss()
# Alegerea optimizatorului (rata de invatare ramane constanta pe toata durata antrenarii)
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay = 0.005, momentum = 0.9)

In [ ]:
# Antrenarea modelului
total_step = len(train_loader)

# Retinem ponderile modelului cu cea mai buna acuratete pe validare
best_acc = 0.0
best_epoch = 0
best_state = copy.deepcopy(model.state_dict())

for epoch in tqdm(range(num_epochs)):
    # Modul de antrenare: dropout activ, BatchNorm actualizeaza statisticile
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        # Incarcam tensorii pe gpu/cpu
        images = images.to(device)
        labels = labels.to(device)

        # Forward propagation
        outputs = model(images) # forward propagation
        loss = criterion(outputs, labels) # calculul functiei de pierdere

        # Backprop si rularea unui pas de optimizare a ponderilor
        optimizer.zero_grad() # resetarea gradientilor (altfel, ei s-ar acumula pe parcursul iteratiilor succesive)
        loss.backward() # calcularea noilor gradienti, aferenti iteratiei curente
        optimizer.step() # actualizarea ponderilor

        running_loss += loss.item()

    # tqdm.write afiseaza mesajul fara sa strice bara de progres
    tqdm.write('Epoch [{}/{}], Steps: {}, Average loss: {:.4f}'
               .format(epoch+1, num_epochs, total_step, running_loss / total_step))

    # Rularea evaluarii:
    model.eval() # setarea algoritmului in modul de evaluare (dropout inactiv, BatchNorm cu statisticile invatate)
    with torch.no_grad(): # nu ne dorim sa mai calculam gradientii, deoarece facem doar forward propagation in timpul evaluarii
        correct = 0
        total = 0
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            del images, labels, outputs

        valid_acc = 100 * correct / total
        tqdm.write('Accuracy of the network on the {} validation images: {} %'.format(total, valid_acc))

    if valid_acc > best_acc:
        best_acc = valid_acc
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())

print('Best validation accuracy: {} % (epoch {})'.format(best_acc, best_epoch))

## 4. Testarea retelei pe `test`

In [ ]:
# Testam modelul cu cea mai buna acuratete pe validare (nu neaparat cel din ultima epoca)
model.load_state_dict(best_state)

# Rularea algoritmului pe baza de test, in modul de evaluare
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        del images, labels, outputs

    print('Accuracy of the network on the {} test images: {} %'.format(total, 100 * correct / total))

# \#TODO:
1. Implementati structura retelei AlexNet. Clasa trebuie sa defineasca straturile in functia \_\_init\_\_() si sa le lege in functia forward().
2. Comparati rezultatele antrenarii cu, respectiv fara, augmentari.
3. Reduceti setul de date de antrenare la 10% din valoarea sa si analizati impactul asupra acuratetii finale.
4. Incercati diferiti optimizatori si hiperparametri. Testarea se ruleaza doar dupa ce se fixeaza toti hiperparametrii si arhitectura de retea.
5. Mariti numarul de epoci si implementati mecanismul de early stopping cu o fereastra de analiza de 3 epoci.